# Invoke a Foundry agent via REST - streaming (SSE)

Extends [`08-09-01-rest-single-shot.ipynb`](08-09-01-rest-single-shot.ipynb) by adding `"stream": true` to the request. The Responses API replies with a `text/event-stream` response - a sequence of Server-Sent Events (SSE), each a small JSON document - rather than one final JSON body. The client parses each event as it arrives, which lets the UI render tokens incrementally instead of waiting for the full answer.

No other 08-agents notebook demonstrates SSE explicitly, so this also serves as the canonical example of the event-type dispatch pattern.

## 1. Setup

Same env, same token scope, same agent. Only the request body and response handling differ.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

AGENT_NAME = 'storytelling-agent'

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

endpoint = os.environ['ALPHA_FOUNDRY_PROJECT_ENDPOINT'].rstrip('/')
responses_url = f'{endpoint}/openai/v1/responses'

credential = DefaultAzureCredential()
access_token = credential.get_token('https://ai.azure.com/.default').token

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json',
    'Accept': 'text/event-stream',
}

print(f'Responses URL: {responses_url}')

## 2. Request body with `stream: true`

Only one field changes from the single-shot body: `stream` is set to `true`. The server's response content type flips from `application/json` to `text/event-stream`.

In [ ]:
body = {
    'input': [
        {'role': 'user', 'content': 'Tell me a three-sentence story about a lighthouse keeper.'}
    ],
    'agent_reference': {
        'name': AGENT_NAME,
        'type': 'agent_reference',
    },
    'stream': True,
}

print(json.dumps(body, indent=2))

## 3. SSE parser

The Server-Sent Events format is line-delimited: each event is one or more `field: value` lines followed by a blank line. The Responses API only uses the `data:` field, and each `data:` value is a complete JSON document. So a minimal parser is:

1. Open the response in streaming mode (`stream=True`).
2. Iterate `iter_lines()` and watch for lines starting with `data: `.
3. Parse the JSON payload after `data: ` and dispatch on its `type` field.

Event types the Responses API emits (non-exhaustive):

| Event type | Meaning |
|------------|---------|
| `response.created` | Stream opened - carries the new `response.id`. |
| `response.output_item.added` | A new output item (e.g. `message`, `function_call`) was added. |
| `response.content_part.added` | A new content part inside an item was added. |
| `response.output_text.delta` | Incremental text chunk for an `output_text` content part. **This is where you accumulate visible tokens.** |
| `response.output_text.done` | A given `output_text` part is complete. |
| `response.completed` | The full response is done - carries the final `response` object. |
| `error` | An error occurred mid-stream. |

In [ ]:
response_id = None
accumulated_text = []
event_counts = {}

with requests.post(responses_url, headers=headers, json=body, stream=True, timeout=120) as resp:
    resp.raise_for_status()
    print(f'HTTP status        : {resp.status_code}')
    print(f'Response content-type: {resp.headers.get("content-type")}')
    print()
    print('Streaming output:')
    print('-' * 60)

    for raw_line in resp.iter_lines(decode_unicode=True):
        if not raw_line or not raw_line.startswith('data: '):
            continue

        payload_str = raw_line[len('data: '):]
        if payload_str == '[DONE]':
            break

        event = json.loads(payload_str)
        event_type = event.get('type', '<no-type>')
        event_counts[event_type] = event_counts.get(event_type, 0) + 1

        if event_type == 'response.created':
            response_id = event.get('response', {}).get('id')
        elif event_type == 'response.output_text.delta':
            delta = event.get('delta', '')
            accumulated_text.append(delta)
            sys.stdout.write(delta)
            sys.stdout.flush()
        elif event_type == 'response.completed':
            pass
        elif event_type == 'error':
            print(f'\n[error] {event}')

    print()
    print('-' * 60)

print()
print(f'Response id      : {response_id}')
print(f'Total characters : {sum(len(s) for s in accumulated_text)}')
print(f'Event counts     : {event_counts}')

## 4. Inspect the assembled text

The accumulated `response.output_text.delta` chunks concatenated together are equivalent to the aggregated `output_text` the single-shot notebook builds from the non-streaming `output` array. (Neither the streaming events nor the final JSON expose a ready-made `output_text` field - that convenience exists only on the SDK's typed `Response`.)</cell id="section-4">

In [ ]:
full_text = ''.join(accumulated_text)
print('Full assembled output:')
print(full_text)

## Summary

| Step | What changes from single-shot |
|------|-------------------------------|
| URL | Same - `{endpoint}/openai/v1/responses` |
| Auth | Same - bearer token, `https://ai.azure.com/.default` |
| Body | Adds `'stream': True` |
| Headers | Adds `Accept: text/event-stream` (optional but explicit) |
| Request call | `requests.post(..., stream=True)` - keep the response open |
| Response | Iterate `resp.iter_lines()`, parse JSON after `data: `, dispatch on `event['type']` |
| Output text | Accumulate `event['delta']` from `response.output_text.delta` events |

### When this matters

Streaming pays off for:

- **Interactive UIs** - render tokens as they arrive, lower perceived latency.
- **Long-running tool loops** - observe `function_call` items and tool progress events without waiting for the full response.
- **Cancellation** - the client can abort an in-progress request before the agent finishes generating.

The trade-off is parser complexity: the JSON-document-per-line response of the non-streaming call becomes an event stream that the client must reassemble.

### Related

- [Foundry SDK streaming](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses) - the SDK exposes `client.responses.stream(...)` which wraps this exact event protocol.